# Week 6 Homework: Your Answers vs. Your Vibe-Coded Dashboard

**Week 4 dashboard link:** https://jing454-ctp-data-graphing-app-lcesdo.streamlit.app/

---

Two weeks ago you vibe-coded a MovieLens dashboard before you had the tools to judge it. Since then you've had the stats lecture (Week 5) and this week's EDA lecture. Now you do it the real way: answer the same 4 questions yourself, by hand, then audit the dashboard against your own work.

**Ground rules**

- 🚫 **No AI for code or answers in Parts 1 and 2.** Lecture notebooks, pandas/seaborn docs, Google, and Stack Overflow are all fair game. Write Part 3 in your own words too.
- 🔒 **Lock in before you peek.** Finish Parts 1 and 2 (answers written down) *before* you open your dashboard. If you look first, you'll anchor on its numbers and the comparison is worthless.
- 🔢 **Show your numbers.** Every answer needs actual values (counts, averages, titles), not just a vibe.
- ✏️ Written answers go in markdown cells (double-click to edit, Shift+Enter to render). Most are 2 to 4 sentences. Short and specific beats long and vague.

## The data

`movie_ratings.csv` has one row per rating: 100,000 ratings from 943 users on 1,682 movies.

- `user_id`, `movie_id`: who rated, which movie
- `rating`: 1 to 5 stars
- `timestamp`, `rating_year`: when the rating was given
- `age`, `gender`, `occupation`, `zip_code`: the rater
- `title`: movie title, with release year in parentheses
- `year`, `decade`: the movie's release year and decade
- `genres`: all of the movie's genres in one string, separated by the pipe character (a movie can have several)

## Setup

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Works whether this notebook sits in the week folder or inside exercise/
data_path = next(p for p in ["data/movie_ratings.csv", "../data/movie_ratings.csv"] if Path(p).exists())
df = pd.read_csv(data_path)
df.head()

,user_id,movie_id,rating,timestamp,age,gender,occupation,zip_code,title,year,decade,genres,rating_year
0,196,242,3,1997-12-04 15:55:49,49,M,writer,55105,Kolya (1996),1996.0,1990.0,Comedy,1997
1,186,302,3,1998-04-04 19:22:22,39,F,executive,00000,L.A. Confidential (1997),1997.0,1990.0,Crime|Film-Noir|Mystery|Thriller,1998
2,22,377,1,1997-11-07 07:18:36,25,M,writer,40206,Heavyweights (1994),1994.0,1990.0,Children|Comedy,1997
3,244,51,2,1997-11-27 05:02:03,28,M,technician,80525,Legends of the Fall (1994),1994.0,1990.0,Drama|Romance|War|Western,1997
4,166,346,1,1998-02-02 05:33:16,47,M,educator,55113,Jackie Brown (1997),1997.0,1990.0,Crime|Drama,1998


---
# Part 1: Methodology

## 1.1 Know your data first

Before answering anything, check what you're working with: shape, column types, missing values, duplicate rows, and the range of the numeric columns.

In [2]:
# Your data checks here
print(f'shape: {df.shape}') 
#shape is 100,000 rows and 13 columns

print(f'{df.dtypes}')

print(df.isnull().sum())
# 30 rows of year and decades are missing 

print(df.describe())

print(df[df.duplicated()])
#no duplicated exist

df[df["year"].isnull()]

print(f'title total nunique: {df.title.nunique()}')
print(f'rating total nunique: {df.rating.nunique()}')


shape: (100000, 13)
user_id          int64
movie_id         int64
rating           int64
timestamp          str
age              int64
gender             str
occupation         str
zip_code           str
title              str
year           float64
decade         float64
genres             str
rating_year      int64
dtype: object
user_id         0
movie_id        0
rating          0
timestamp       0
age             0
gender          0
occupation      0
zip_code        0
title           0
year           30
decade         30
genres          0
rating_year     0
dtype: int64
            user_id       movie_id         rating            age  \
count  100000.00000  100000.000000  100000.000000  100000.000000   
mean      462.48475     425.530130       3.529860      32.969850   
std       266.61442     330.798356       1.125674      11.562623   
min         1.00000       1.000000       1.000000       7.000000   
25%       254.00000     175.000000       3.000000      24.000000   
50%       44

**✏️ Data notes:** _(What did you find? Anything missing, duplicated, or surprising? What did you do about it, and why?)_

30 Null values on year and decade. 
Some reviews has unknown title of the movie they're reviewing
Generally the year and decade are unknown together
rating is int (generally should be floats)
year and decade are floats. There are no such thing as float years (should be int)

## 1.2 Decide before you chart

In Week 4 the questions were ambiguous on purpose. This is where *you* commit to an interpretation. Fill it in now, and update it if you change your mind while working (just say so).

**Q1 Genre breakdown**
- What you have to decide: What am I counting, ratings or movies? How do I handle movies that have several genres?
- My choice: I decided to count ratings to work with. My options for the genre could be just to use the primary genres instead of multiple that were listed.
- Why: Rating are easier to work with as there are only 4 different rating to work with and where as title has way way more (1664 different to be exact). Additionally, i feel like we can work on more analysis with other columns compared to movie titles. Converting generes to only their primary genre would be nice too as it should state its primary genre (although it may lose its context a little)

**Q2 Genre satisfaction**
- What you have to decide: "Average rating" of what? How do I treat genres that have very little data?
- My choice: Perhaps the average rating of a "movie" or the average rating of a genre. Converting genres into its primary genre solves the issue of genres that has little data. If there exist a primary genre where it truely has little to little no data. Additionally, you may be able to break up the genres into several entries for accuracy
- Why: Generally getting the rating of a genre or movie title would probably be the most important and common analysis that we're trying to get. 

**Q3 Ratings over time**
- What you have to decide: Is every year backed by the same amount of data? What do I do with movies that have no release year?
- My choice: Extremely unlikely. Generally data like this will almost never be balanced between the years. Generally if you're doing analysis on the release year, you would want to probably drop it to prevent data from being skewed. If the data is < 5% of total data then you can drop the data
- Why: Code would be ran. Data is compeltely random and it's impossible to get anything equal unless something was done to it. As for the unknown year, if it's less than 5% of the data, dropping the unknown years wont really affect the data as there is too little data for it to affect the analysis. 5% is the stopping point. (anything over 5% shouldn't be just dropped as there are other useful data that could be extracted from the unknown years)

**Q4 Best movies**
- What you have to decide: What is a movie's rating (the mean of what)? Is "at least 50" ≥ or >? What if there's a tie?
- My choice: It depends. You can compare movie's rating to genre and see the mean of the rating for each genre or something. you can only compared it to the year, and even to the title. At least 50 would be >=. If there's a tie, we could even rank them based on the more total entries or rank them the same
- Why: Rating is very flexible and could be drawn alot of analysis with and every one of them are important and insightful. "At least" means if it reaches that threshold hence 50 >=. Ties won't really affect the data that much so likely nothing to worry about. However, if we want something done with it, then we could put whatever has longer total up in the found..

---
# Part 2: Results (by hand, no AI)

Answer each question with a chart or table **and** a written answer with real numbers. Pick the chart type yourself and be ready to defend it in Part 3. Add as many code cells as you need.

## Q1. Genre breakdown

**What's the distribution of genres among the movies that were rated?**

<details>
<summary>Stuck on movies with several genres? Hint</summary>

`genres` is one string like `Crime|Film-Noir|Mystery|Thriller`. To get one row per genre:

```python
genre_df = df.assign(genre=df['genres'].str.split('|')).explode('genre')
```

Heads up: after this, a movie with 3 genres takes up 3 rows. Think about what that does to anything you count or average. Exploding is fine for "how often does a genre show up", but be careful calling the result "market share" (the percentages add up to more than 100%).
</details>

In [3]:
# Q1: your code here

genre_df = df.assign(genre=df['genres'].str.split("|")).explode("genre")

genre_df.head()

genre_df.genre.value_counts(normalize=True)

genre
Drama          0.187657
Comedy         0.140323
Action         0.120365
Thriller       0.102881
Romance        0.091540
Adventure      0.064691
Sci-Fi         0.059879
War            0.044206
Crime          0.037889
Children       0.033783
Horror         0.025010
Mystery        0.024671
Musical        0.023303
Animation      0.016957
Western        0.008721
Film-Noir      0.008152
Fantasy        0.006360
Documentary    0.003565
unknown        0.000047
Name: proportion, dtype: float64

**✏️ My answer (with numbers):** _(Which genres dominate, by how much, and what exactly did you count?)_
Drama makes up 18.7% of the total genre where as the 2nd highest is comedy at 14% which is 4% below the highest. Value_counts counted all data in a percentage way (including unknown)

## Q2. Genre satisfaction

**Which genres have the highest average rating? Which have the lowest?**

Show how much data is behind each average (for example, the count next to it). An average of 10 ratings and an average of 10,000 ratings are not the same kind of evidence.

In [4]:
# Q2: your code here
genre_df.groupby("genre")["rating"].mean().sort_values(ascending=False)

genre_df.groupby("genre")["rating"].agg(rating_avg= "mean", rating_count="sum").sort_values(by="rating_avg", ascending=False).head(10)

,rating_avg,rating_count
genre,,
Film-Noir,3.921523,6796
War,3.815812,35861
Drama,3.687379,147108
Documentary,3.672823,2784
Mystery,3.638132,19082
Crime,3.632278,29258
Romance,3.621705,70482
Western,3.613269,6699
Animation,3.576699,12894


In [5]:
(genre_df["genre"] == "Film-Noir").sum()
(genre_df["genre"] == "Fantasy").sum()

np.int64(1352)

**✏️ My answer (with numbers):** _(Top and bottom genres, their averages, and how much data each one rests on.)_
Top rated genre is Film-Noir, coming in at 3.92 average rating with 1733 total movies. The bottom real genre is Fantasy, coming in at 3.21 average rating with 1352 total movies

## Q3. Ratings over time

**How has the mean rating changed across movie release years?**

Show how many ratings each year is based on, alongside the trend.

In [6]:
# Q3: your code here
genre_df.year.value_counts()

print(f'Size of DataFrame: {len(genre_df)}')
genre_df.groupby("year")["rating"].mean().sort_index(ascending=False).head(70)


Size of DataFrame: 212595


year
1998.0    3.323529
1997.0    3.325326
1996.0    3.312172
1995.0    3.443458
1994.0    3.426577
            ...   
1933.0    3.983957
1932.0    3.833333
1931.0    4.000000
1930.0    3.777778
1926.0    3.000000
Name: rating, Length: 70, dtype: float64

**✏️ My answer (with numbers):** _(What's the trend? Is it equally believable across the whole timeline?)_

It seems like as the years goes on, the overall rating goes lower. However, this isnt accurate over the last 70 years as you can see the decline in average rating during the 1926 as well but just not as large. Therefore, this observation is very shakely

We can observe an average rating of 3.323 on 1998 and 3.4 on 1994. However, then you can observe how during 1926, it went to 3.0 and was going inbetween up and down during 1930-1933 which shows inconsistant data.


## Q4. Best movies, with a floor

**What are the 5 best-rated movies that have at least 50 ratings? What changes if you raise the floor to 150?**

In [7]:
# Q4: your code here

genre_df.groupby("title")["rating"].agg(rating_avg= "mean", rating_count="sum").loc[lambda x: x["rating_count"] >= 50].sort_values(by="rating_avg", ascending=False).head(5)

#genre_df.groupby("title")["rating"].agg(rating_avg= "mean", rating_count="sum").loc[lambda x: x["rating_count"] >= 150].sort_values(by="rating_avg", ascending=False).head(5)


,rating_avg,rating_count
title,,
Star Kid (1997),5.000000,60
"Close Shave, A (1995)",4.491071,1509
Schindler's List (1993),4.466443,2662
"Wrong Trousers, The (1993)",4.466102,1054
Casablanca (1942),4.456790,3249


**✏️ My answer (with numbers):** _(Both top-5 lists. Which movies dropped out or moved in when the floor went up, and what does that say about small samples?)_

Top 5 ranked Movies (>= 50 ratings):

Star Kid (1997): 5.0
Close Shave, A (1995): 4.49
Schindler's List (1993): 4.46
Wrong Trousers, The (1993): 4.46
Casablanca (1942): 4.45

Top 5 ranked Movies (>= 150 ratings)

Close Shave, A (1995): 4.49 
Schindler's List (1993): 4.46
Wrong Trousers, The (1993): 4.46
Casabalanca (1942): 4.45
Wallace & Gromit: The Best of Aardman Animation (1996): 4.44

Star kid was dropped and Wallace & Gromit came into the top 5 list (from >= 50 filter to >= 150). Small samples shouldn't be as important compared to bigger samples since they have lower # of rating (not as detailed for analysis since changed too easily)

> 🔒 **Stop here.** Parts 1 and 2 should be completely filled in before you open your dashboard.

---
# Part 3: Analysis

Now open your Week 4 dashboard.

## 3.1 How the dashboard got built

Paste 2 prompts that shaped it (from your build log, or from your AI chat history if you didn't keep a log; don't invent any). For each one: what did you ask, what did the AI give you first, and what did you change or accept without checking?

**Prompt 1**
- What I asked:
[requirements.txt](requirements.txt) make sure to follow this file's requirements
[app.py](app.py) put the code in app.py
make sure the graphs are seperate.
here are the questions provided so you can divide it to the different charts.
Question 1 — Genre Breakdown: What's the distribution of genres among the movies that were rated? (Movies can have multiple genres — have the AI explain how it handled that before it counts anything.)
Question 2 — Genre Satisfaction: Which genres have the highest average rating? Which have the lowest?
Question 3 — Ratings Over Time: How has the mean rating changed across movie release years?
Question 4 — Best Movies, With a Floor: What are the top 5 best-rated movies, once you only count movies with at least 50 ratings? What changes if you raise that floor to 150?
here's the data: 
Genre	Movies	Share of rated movies
Drama	725	43.1%
Comedy	505	30.0%
Thriller	251	14.9%
Action	251	14.9%
Romance	247	14.7%
Adventure	135	8.0%
Children	122	7.3%
Crime	109	6.5%
Sci-Fi	101	6.0%
Horror	92	5.5%
War	71	4.2%
Mystery	61	3.6%
Musical	56	3.3%
Documentary	50	3.0%
Animation	42	2.5%
Western	27	1.6%
Film-Noir	24	1.4%
Fantasy	22	1.3%
Unknown / blank	2	0.1%


Film-Noir has the highest average rating at 3.92/5 (1,733 ratings), followed by War at 3.82/5 (9,398 ratings).
The lowest is Fantasy at 3.22/5 (1,352 ratings). There are also 10 ratings with an unknown or blank genre, averaging 3.20; excluding those, Fantasy is the lowest.
Mean ratings vary by release year, but the clearer pattern is a dip in the newest years represented in the file: averages were about 3.9–4.0 for many years from the 1930s through the 1980s, then fell to 3.58 in 1990 and 3.31 by 1996–1998. Ratings for 1993–1997 are each based on many thousands of ratings; 1998 has only 851, so its average is less stable. 
With at least 50 ratings, the top five are:
Rank	Movie	Mean rating	Ratings
1	A Close Shave (1995)	4.491	112
2	Schindler’s List (1993)	4.466	298
3	The Wrong Trousers (1993)	4.466	118
4	Casablanca (1942)	4.457	243
5	Wallace & Gromit: The Best of Aardman Animation (1996)	4.448	67


With at least 150 ratings, the top five become:
Rank	Movie	Mean rating	Ratings
1	Schindler’s List (1993)	4.466	298
2	Casablanca (1942)	4.457	243
3	The Shawshank Redemption (1994)	4.445	283
4	Rear Window (1954)	4.388	209
5	The Usual Suspects (1995)	4.386	267

- AI's first attempt:Created a dashboard based on the data. Color scheme is incorrect, making the charts and graphs very ugly
- What I changed (or accepted without checking): Ended up checking the code to ensure code looks right and it's using the right data

**Prompt 2**
- What I asked:[app.py](app.py) take a look at it. the rating over time graph chart are way too close together, forming a huge block
use the following graphs however you feel is the best.
Bar chart	Comparing counts or averages across categories	Ask for it sorted — don't accept default/alphabetical order
Horizontal bar chart	Same, but with long labels or many categories	Still needs sorting; watch for too many bars to read at a glance
Pie chart	Parts of a whole, only with a handful of categories	AI loves defaulting to pie charts for things like "genre breakdown" with 18 slices — notice if it does this
Line / time series chart	Trends over time	Notice if the AI confuses "year released" with "year rated"
Scatter plot	Relationship between two continuous variables	With thousands of points, ask for transparency/aggregation to avoid overplotting
Heatmap	Two categorical dimensions against a value	Needs a legend/color scale to be r
- AI's first attempt: Ended up fixing the line plot graph. Although still weird to to color scheme
- What I changed (or accepted without checking):Checked the code to ensure nothing weird is added then ran it.

## 3.2 Compare the answers

Read the answer off your dashboard for each question and put it next to yours.

**Q1**
- Dashboard says: 
Drama(43.1%), 
Comedy(30%), 
Thriller(14.9%),
Action(14.9%), 
Romance(14.7%), 
Adventure(8%), 
Children(7.3%), 
Crime(6.5%), 
Sci-Fi(6%), 
Horror(5.5%), 
War(4.2%),
Mystery(3.6%), 
Musical(3.3%), 
Documentary(3.0%), 
Animation(2.5%),
Western(1.6%), 
Film-Noir(1.4%),
Fantasy(1.3%),
Unknown(0.1%)

- I say:
Drama          0.187657
Comedy         0.140323
Action         0.120365
Thriller       0.102881
Romance        0.091540
Adventure      0.064691
Sci-Fi         0.059879
War            0.044206
Crime          0.037889
Children       0.033783
Horror         0.025010
Mystery        0.024671
Musical        0.023303
Animation      0.016957
Western        0.008721
Film-Noir      0.008152
Fantasy        0.006360
Documentary    0.003565
unknown        0.000047

- Same? (yes / partly / no): partly

- If different, why?: The explode method causes the dataset to have inflated certain genre and their total number of movies for each genre

**Q2**
- Dashboard says:
Film-Noir: 3.92
War: 3.82
Fantasy: 3.22
Uknown: 3.2

- I say:

Film-Noir	3.921523	6796
War	3.815812	35861
Drama	3.687379	147108
Documentary	3.672823	2784
Mystery	3.638132	19082
Crime	3.632278	29258
Romance	3.621705	70482
Western	3.613269	6699
Animation	3.576699	12894
Sci-Fi	3.560723	45328

- Same? (yes / partly / no): no

- If different, why?: Most likely because of the fact that when we exploded the genre, multiple movie with different ratings affected the dataset and altered the rating score of different genres


**Q3**
- Dashboard says:
1930 starts at 3.9 and slowly climbs to 4 at 1980. It then drops to 3.58 in 1990 (deep slope) and continue to drop slowly to 3.31 till 1998

- I say:

year
1998.0    3.323529
1997.0    3.325326
1996.0    3.312172
1995.0    3.443458
1994.0    3.426577
            ...   
1933.0    3.983957
1932.0    3.833333
1931.0    4.000000
1930.0    3.777778
1926.0    3.000000


- Same? (yes / partly / no):partly
- If different, why?: Could be due to the fact that some of these years had unknown which altered the data. Another reason could be because there are more entries

**Q4**
- Dashboard says:

Minimum 50 ratings:
title                                                   rating_avg   rating_count
A Close Shave (1995)                                    4.491        112
Schindler's List (1993)                                 4.466        298
The Wrong Trousers (1993)                               4.466        118
Casablaca (1942)                                        4.457        243
Wallace & Gromit: The Best of Aardman Animation (1996)  4.448        67

Minimum 150 ratings:
title                                                   rating_avg  rating_count
Schindler's list (1993)                                 4.466       298
Casablanca (1942)                                       4.457       243
The Shawshank Redemption (1994)                         4.445       283
Rear Window (1954)                                      4.388       209
The Usual Suspects (1995)                               4.386       267

- I say:

Minimum 50 ratings:
title                                                   rating_avg	rating_count		
Star Kid (1997)	                                        5.000000	60
Close Shave, A (1995)	                                4.491071	1509
Schindler's List (1993)	                                4.466443	2662
Wrong Trousers, The (1993)	                            4.466102	1054
Casablanca (1942)	                                    4.456790	3249

Minimum 150 ratings:
Title                                                   rating_avg	rating_count
Close Shave, A (1995)	                                4.491071	1509
Schindler's List (1993)	                                4.466443	2662
Wrong Trousers, The (1993)	                            4.466102	1054
Casablanca (1942)	                                    4.456790	3249
Wallace & Gromit: The Best of Aardman Animation (1996)	4.447761	298

- Same? (yes / partly / no): partly
- If different, why?: Most likely due to the number of ratings that each movie recieved. The dataset we have right now have way more entries hence the higher rating_count. Data could also be different since "Star kid" is ranked in the previous chart whereas we do not have it now

**When numbers differ, check these first.** Most mismatches come from one of these:

- Counting **ratings** vs. counting **movies**
- One row per rating vs. **one row per rating per genre** (after exploding)
- **Mean of all ratings** vs. the mean of each movie's average
- "At least 50" as ≥ vs. >, or applying the floor before vs. after another step
- What happened to movies with a **missing year**
- Genres with **tiny sample sizes** left in or filtered out

**✏️ Pick two questions and explain what's going on** (2 to 3 sentences each). Track down the *actual cause*: read your dashboard's code, or re-run it with one change. "The AI did it differently" is not an explanation. If the answers match, say why you now trust them.

1. Multiple rows of the same movie split from .str.split().explode() caused the inflated of data counts which threw off some of the part of the analysis (like part 2)
2. The str.replace().explode() also affected the rating average of the analysis. Since we had extra entries of the same movie, that meant that we had extra counts of the same year and rating which affected the analysis from part 3 and part 1

## 3.3 Chart choices

**Q1**
- Chart the dashboard used: bar plot
- Chart I used: bar plot
- Which communicates the answer better, and why?: It works well since there are many entires. other charts like the pie chart wouldn't be so good

**Q2**
- Chart the dashboard used:bar plot
- Chart I used: pie chart
- Which communicates the answer better, and why?: It just feels alot more pleasing to look at and it shows the data better

**Q3**
- Chart the dashboard used: line plot
- Chart I used: line plot
- Which communicates the answer better, and why?: Perfect for timeline analysis and seeing how the trend changes

**Q4**
- Chart the dashboard used: bar plots
- Chart I used: pie chart
- Which communicates the answer better, and why?: Pie chart since only need the top 5 which would work better with a pie chart

Use what you learned about chart choice and misleading charts (Week 5) and this week's lecture. Is it sorted? Are the labels readable? Are the axes honest? Does it show how much data is behind each number?

The charts are sorted. labels are readable. The axes are labeled and clear. The chart also not only the percentage but also the count of the total number of ratings.

In [8]:
df.groupby("user_id").size().sort_values(ascending=False)

#df.groupby("user_id").size().median()


user_id
405    737
655    685
13     636
450    540
276    518
      ... 
888     20
572     20
571     20
558     20
895     20
Length: 943, dtype: int64

In [9]:
df["rating_year"].value_counts()

rating_year
1997    52899
1998    47101
Name: count, dtype: int64

## 3.4 What can this data actually tell you?

**Answer at least 3 of these 5**, using your own results as evidence.

**1. Who is in this data?** Check the fewest and the most ratings any single user has given (`df.groupby('user_id').size()`), and look at `rating_year`. What kind of person ends up in a dataset like this, and who is missing? How might that tilt your Q2 answer? *(Week 5: sampling bias)*


**2. How sure are you about the bottom genre?** Take the lowest-rated genre from your Q2. How many ratings *and* how many distinct movies is it built on? Estimate a rough 95% interval: mean ± 1.96 × std / √n. Would you bet it's truly the worst genre? *(Week 5: standard error and confidence intervals. Also ask yourself: are 5,000 ratings of the same handful of movies really 5,000 independent pieces of evidence? Remember the one-row-per-model lesson from last week's homework.)*

**3. What can Q3 really say about time?** How many different values does `rating_year` have? So can this data tell you whether people's taste changed over time? What does your Q3 trend actually describe instead? Why might an old movie in this dataset score higher than a recent one? *(Think: who rates a 1940s film in 1997, and which old movies are even in the data?)*

**4. Does "average" depend on how you average?** Recompute Q2 two ways: (a) average over all ratings, (b) average each movie first, then average the movies within each genre. Does the top genre change? "Which genre do *people* like most?" and "which genre has the best *movies*?" are different questions. *(Week 5: confounding, Simpson's paradox)*

**5. Deployment risk.** A studio exec uses your dashboard to decide which genre to green-light. What's the most misleading conclusion they could draw from one of your charts? Write the one-line caveat you'd put under that chart.


**✏️ Your answers:** _(label each one with its number)_

1. Answer: If you take a look at the number of ratings made by a person, the most ratings made by a person is 737 times by "user 405" and the lowest is 20 times. However, if you take a look at the median, it is only 45. This shows perhaps a sampling bias since this only contains the most active users for rating whereas others barely rates. This causes a sampling bias due to a person favoring one demographic more than another. As a result, this ineffectives affects Q2 which is on genre. 

3. Answer: There only exists 2 values in `rating_year`. We are unable to check how people's taste changed this these rating were only done between 1997-1998. So, in reality we were just getting people's rating from 1997-1998 and not the other times. Additionally, older movies most likely were scored highly due to the restricted seperation of users that rated because those user liked classic movies (which couwld be possible)

5. Answer: Since this data is mainly favors towards the people that usual do these ratings the most, these data may not be accurate at all due to sampling bias. Since the data is completely restricted towards one party, the public opinions were not put into the analysis.

## 3.5 Verdict

**Where I'd override the AI** (something it got wrong, hid, or did poorly, and what your version did better): Some of its wording were rewritten in a different way that better fitted the analysis

**Where the AI held up** (something it did well, or a choice you doubted at first that turned out right):
Suprisingly it was actually pretty decent at analysis. It got most of the things correctly (after checking)

**One sentence:** looking back at both weeks, who was driving, you or the AI, and when? Previously, the AI was driving. Now, the AI is also still kinda driving but required human for reviewal and approval

---
## Optional stretch

Only if you want more practice:

- Pick 4 genres. How does the mean rating change as viewer age increases? (Try more than 4 and see which genres make the most interesting chart.)
- Plot number of ratings vs. mean rating for each movie. Do the most-rated movies get higher ratings? Is there a correlation? *(Week 5: correlation vs. causation)*

---
## Before you submit

- [ ] Copy this notebook into the `exercise/` folder and rename it `Firstname_Lastname_week_06_exercise.ipynb`
- [ ] **Kernel → Restart & Run All** and make sure it runs top to bottom with your charts showing
- [ ] Every ✏️ cell and every fill-in line is filled in
- [ ] Open a PR from your fork, same as every other week